<a href="https://colab.research.google.com/github/nageshsarode52/DistilGPT_Fine_Tuning_Retail_-_E_commerce.ipynb/blob/main/DistilGPT_Fine_Tuning_Retail_%26_E_commerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit transformers datasets pyngrok accelerate

In [ ]:
from datasets import load_dataset

dataset = load_dataset("amazon_polarity")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 3600000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 400000
    })
})
{'label': 1, 'title': 'Stuning even for the non-gamer', 'content': 'This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^'}


In [ ]:

train_data = []

for item in dataset["train"].select(range(500)):

    review = item["content"]   # correct for amazon_polarity
    label = item["label"]

    if label == 1:
        response = "Thanks for your positive review! We are happy you liked the product."
    else:
        response = "We are sorry for your experience. We will improve our service."

    train_data.append({
        "text": f"User: {review}\nAssistant: {response}"
    })

print(train_data[0])

{'text': 'User: This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^\nAssistant: Thanks for your positive review! We are happy you liked the product.'}


In [ ]:
import json

with open("dataset.json", "w") as f:
    json.dump(train_data, f)

print("Dataset ready ✔")

Dataset ready ✔


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset

model_name = "distilgpt2"

dataset = load_dataset("json", data_files="dataset.json")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

#  FIX: add labels (THIS WAS MISSING)
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()   #  IMPORTANT FIX
    return tokens

tokenized = dataset.map(tokenize)

model = AutoModelForCausalLM.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir="./shopbot_model",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=10,
    save_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"]
)

trainer.train()

model.save_pretrained("./shopbot_model")
tokenizer.save_pretrained("./shopbot_model")

print("Training completed ✔")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.229871
20,3.015942
30,2.808037
40,2.984882
50,2.745641
60,2.515406
70,2.987079
80,2.883598
90,2.637467
100,2.436750


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed ✔


In [ ]:
!pip install gradio -q
!pip install transformers -q

import gradio as gr
from transformers import pipeline

# lightweight model (safe for Colab)
pipe = pipeline("text-generation", model="distilgpt2")

def chat(user_input):
    output = pipe(user_input, max_new_tokens=80)
    return output[0]["generated_text"]

ui = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Ask your question"),
    outputs=gr.Textbox(label="ShopBot Response"),
    title="🛍️ Retail & E-commerce AI Chatbot"
)

ui.launch(share=True)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://091fae375bd14807e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
